# Jaw Click Decoder: Turning a Reliable Biosignal into a Control Input

## Section 1 — Introduction

Jaw clench activity produces a strong biosignal relative to subtle left/right EEG intent. In this project, jaw activity became the most reliable discrete control signal for a click-style action.

This notebook shows the full professor-facing workflow from uploaded OpenBCI files to a usable click decoder:

- raw signal inspection
- preprocessing for jaw-focused decoding
- window extraction and labeling
- interpretable feature engineering
- model comparison and evaluation
- probability-to-click replay logic

The goal is not only classification accuracy. The deeper question is whether classifier confidence can be turned into a stable, real-time control input.


In [ ]:
import io
import json
import pickle
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, welch
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from google.colab import files
from IPython.display import Markdown, display

plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "figure.dpi": 120,
        "axes.facecolor": "#fbfbfd",
        "figure.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 14,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.frameon": False,
    }
)

DEFAULT_FS_FALLBACK = 250.0
CLASS_ORDER = ["REST", "CLENCH"]
SIGNAL_NAME_RE = re.compile(r"^(ch(?:annel)?|eeg|emg|exg|adc)[ _-]*\d+$", re.IGNORECASE)
INDEX_HINTS = ("sample", "index")
TIME_HINTS = ("time", "timestamp", "ts")
MARKER_HINTS = ("marker", "event", "trigger", "label", "stim", "class")


## Section 2 — Upload Files

Upload one or more OpenBCI files directly into Colab. The notebook will:

- detect CSV-like recordings for jaw analysis
- separately list optional artifacts such as `.pkl`, `.json`, and `.md`
- let you choose which uploaded CSV files should be included in the decoder workflow

This keeps the notebook flexible for jaw-only sessions, LRJ hybrid sessions with JAW markers, and optional supporting files.


In [ ]:
print("Upload OpenBCI recordings and any optional supporting artifacts.")
uploaded_raw = files.upload()

if not uploaded_raw:
    raise RuntimeError("No files were uploaded. Re-run this cell and choose one or more files.")

UPLOADED_FILES = {name: blob for name, blob in uploaded_raw.items()}
UPLOADED_FILENAMES = sorted(UPLOADED_FILES.keys())

CSV_EXTENSIONS = (".csv", ".txt", ".tsv")
ARTIFACT_EXTENSIONS = (".pkl", ".pickle", ".json", ".md", ".txt")

CSV_FILES = [name for name in UPLOADED_FILENAMES if name.lower().endswith(CSV_EXTENSIONS)]
OPTIONAL_ARTIFACT_FILES = [
    name for name in UPLOADED_FILENAMES
    if name not in CSV_FILES and name.lower().endswith(ARTIFACT_EXTENSIONS)
]
OTHER_FILES = [name for name in UPLOADED_FILENAMES if name not in CSV_FILES and name not in OPTIONAL_ARTIFACT_FILES]

upload_table = pd.DataFrame(
    {
        "Index": np.arange(1, len(UPLOADED_FILENAMES) + 1),
        "Filename": UPLOADED_FILENAMES,
        "Type": [
            "CSV recording" if name in CSV_FILES else "Optional artifact" if name in OPTIONAL_ARTIFACT_FILES else "Other"
            for name in UPLOADED_FILENAMES
        ],
        "Size (KB)": [round(len(UPLOADED_FILES[name]) / 1024.0, 1) for name in UPLOADED_FILENAMES],
    }
)
display(upload_table)

if not CSV_FILES:
    raise RuntimeError("No CSV-like recordings were detected. Please upload at least one OpenBCI CSV or TSV file.")

print("Detected CSV recordings:")
display(pd.DataFrame({"Index": np.arange(1, len(CSV_FILES) + 1), "Filename": CSV_FILES}))

if OPTIONAL_ARTIFACT_FILES:
    print("Optional uploaded artifacts:")
    display(pd.DataFrame({"Filename": OPTIONAL_ARTIFACT_FILES}))

if OTHER_FILES:
    print("Other uploaded files (not used automatically):")
    display(pd.DataFrame({"Filename": OTHER_FILES}))


def parse_csv_selection(choice_text, available_files):
    if not choice_text.strip():
        return list(available_files)

    selected = []
    pieces = [piece.strip() for piece in choice_text.split(",") if piece.strip()]
    for piece in pieces:
        if piece.isdigit():
            idx = int(piece) - 1
            if idx < 0 or idx >= len(available_files):
                raise IndexError(f"Selection {piece} is outside the CSV file list.")
            selected.append(available_files[idx])
        else:
            if piece not in available_files:
                raise KeyError(f"'{piece}' was not found among the uploaded CSV files.")
            selected.append(piece)
    deduped = []
    for name in selected:
        if name not in deduped:
            deduped.append(name)
    return deduped


selection_prompt = "Enter CSV indices or exact filenames to include in jaw analysis [default: all]: "
selection_text = input(selection_prompt)
SELECTED_CSV_FILES = parse_csv_selection(selection_text, CSV_FILES)

print("Selected CSV files for jaw analysis:")
display(pd.DataFrame({"Filename": SELECTED_CSV_FILES}))


## Section 3 — Robust CSV Parsing

OpenBCI exports can vary across sessions. The parser below is designed to tolerate:

- header rows or no header rows
- named channels such as `ch1` through `ch8`
- unnamed numeric columns
- timestamp or sample-index columns
- marker/event columns at the end
- hybrid files that mix EEG-style and EMG-style content

For each uploaded CSV, the notebook auto-detects likely signal channels, marker columns, and sampling rate. If a sampling rate cannot be inferred reliably, it falls back to **250 Hz**.


In [ ]:
def clean_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def infer_session_family(filename):
    upper_name = filename.upper()
    if "LRJ" in upper_name or "HYBRID" in upper_name:
        return "hybrid"
    if "JAW" in upper_name or "CLENCH" in upper_name or "HR" in upper_name:
        return "jaw"
    if "REST" in upper_name:
        return "rest"
    if "LR" in upper_name:
        return "left_right"
    return "unknown"


def guess_delimiter(lines, sample_count=25):
    candidates = [",", "	", ";"]
    scores = {}
    sample = lines[:sample_count]
    for delimiter in candidates:
        scores[delimiter] = sum(line.count(delimiter) for line in sample)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else ","


def looks_like_header(tokens):
    joined = " ".join(token.strip() for token in tokens)
    return any(char.isalpha() for char in joined)


def find_data_start(lines, delimiter):
    for idx, line in enumerate(lines[:80]):
        tokens = [token.strip() for token in line.split(delimiter)]
        if len(tokens) < 4:
            continue
        numeric_like = 0
        alpha_like = 0
        for token in tokens:
            if token == "":
                continue
            try:
                float(token)
                numeric_like += 1
            except ValueError:
                if any(char.isalpha() for char in token):
                    alpha_like += 1
        if numeric_like >= max(4, len(tokens) // 2):
            return idx
        if alpha_like >= 2 and idx + 1 < len(lines):
            next_tokens = [token.strip() for token in lines[idx + 1].split(delimiter)]
            next_numeric_like = 0
            for token in next_tokens:
                try:
                    float(token)
                    next_numeric_like += 1
                except ValueError:
                    continue
            if next_numeric_like >= max(4, len(next_tokens) // 2):
                return idx + 1
    return 0


def make_unique_columns(columns):
    counts = Counter()
    unique_columns = []
    for idx, name in enumerate(columns):
        candidate = str(name).strip()
        if candidate == "" or candidate.lower().startswith("unnamed"):
            candidate = f"col_{idx}"
        if candidate in counts:
            counts[candidate] += 1
            candidate = f"{candidate}_{counts[candidate]}"
        else:
            counts[candidate] = 0
        unique_columns.append(candidate)
    return unique_columns


def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def estimate_fs_from_time_series(time_series):
    numeric = clean_numeric(time_series)
    finite = numeric.dropna().to_numpy(dtype=float)
    if len(finite) < 8:
        return None, None, None

    diffs = np.diff(finite)
    diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
    if len(diffs) < 5:
        return None, None, None

    median_dt = float(np.median(diffs))
    if not np.isfinite(median_dt) or median_dt <= 0:
        return None, None, None

    fs_seconds = 1.0 / median_dt
    fs_milliseconds = 1000.0 / median_dt

    valid_index = np.flatnonzero(numeric.notna().to_numpy())
    interpolated = np.interp(np.arange(len(numeric)), valid_index, finite)

    if 20 <= fs_seconds <= 5000:
        time_axis_sec = interpolated - interpolated[0]
        return float(fs_seconds), time_axis_sec, "time column interpreted as seconds"

    if 20 <= fs_milliseconds <= 5000:
        time_axis_sec = (interpolated - interpolated[0]) / 1000.0
        return float(fs_milliseconds), time_axis_sec, "time column interpreted as milliseconds"

    return None, None, None


def detect_columns(df):
    normalized = {column: normalize_column_name(column) for column in df.columns}

    numeric_columns = []
    for column in df.columns:
        numeric_series = clean_numeric(df[column])
        if numeric_series.notna().mean() >= 0.80:
            numeric_columns.append(column)

    index_candidates = []
    time_candidates = []
    for column in df.columns:
        column_name = normalized[column]
        if any(hint in column_name for hint in INDEX_HINTS):
            index_candidates.append(column)
        if any(hint in column_name for hint in TIME_HINTS):
            time_candidates.append(column)

    for column in numeric_columns:
        numeric_series = clean_numeric(df[column]).dropna()
        if len(numeric_series) < 10:
            continue
        values = numeric_series.to_numpy(dtype=float)
        diffs = np.diff(values)
        if len(diffs) == 0:
            continue
        positive_ratio = float(np.mean(diffs > 0))
        if positive_ratio < 0.90:
            continue
        median_dt = float(np.median(np.abs(diffs)))
        if not np.isfinite(median_dt) or median_dt <= 0:
            continue
        fs_seconds = 1.0 / median_dt
        fs_milliseconds = 1000.0 / median_dt
        if 20 <= fs_seconds <= 5000 or 20 <= fs_milliseconds <= 5000:
            if column not in time_candidates:
                time_candidates.append(column)
        elif column not in index_candidates:
            index_candidates.append(column)

    marker_column = None
    marker_name_candidates = [
        column for column in df.columns if any(hint in normalized[column] for hint in MARKER_HINTS)
    ]
    if marker_name_candidates:
        marker_column = marker_name_candidates[0]
    else:
        low_cardinality_candidates = []
        for column in reversed(list(df.columns)):
            if column in index_candidates or column in time_candidates:
                continue
            text_series = df[column].dropna().astype(str).str.strip()
            if text_series.empty:
                continue
            unique_count = text_series.nunique()
            if unique_count <= 20:
                low_cardinality_candidates.append((column, unique_count))
        if low_cardinality_candidates:
            marker_column = low_cardinality_candidates[0][0]

    signal_candidates = [column for column in df.columns if SIGNAL_NAME_RE.match(normalized[column])]
    if not signal_candidates:
        excluded = set(index_candidates + time_candidates + ([marker_column] if marker_column else []))
        ranked_numeric = []
        for column in df.columns:
            if column in excluded:
                continue
            numeric_series = clean_numeric(df[column])
            valid_fraction = float(numeric_series.notna().mean())
            unique_count = int(numeric_series.dropna().nunique())
            if valid_fraction >= 0.80 and unique_count >= 25:
                ranked_numeric.append((column, valid_fraction, unique_count))
        if not ranked_numeric:
            for column in df.columns:
                if column in excluded:
                    continue
                numeric_series = clean_numeric(df[column])
                ranked_numeric.append((column, float(numeric_series.notna().mean()), int(numeric_series.dropna().nunique())))
        signal_candidates = [column for column, _, _ in ranked_numeric[:8]]

    excluded = set(index_candidates + time_candidates + ([marker_column] if marker_column else []))
    signal_candidates = [column for column in signal_candidates if column not in excluded][:8]

    return {
        "index_candidates": index_candidates,
        "time_candidates": time_candidates,
        "signal_candidates": signal_candidates,
        "marker_column": marker_column,
    }


def load_openbci_csv(file_bytes, filename):
    text = file_bytes.decode("utf-8", errors="ignore")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [line for line in text.split("\n") if line.strip()]
    if not lines:
        raise ValueError(f"{filename} appears to be empty.")

    delimiter = guess_delimiter(lines)
    data_start_idx = find_data_start(lines, delimiter)
    data_tokens = [token.strip() for token in lines[data_start_idx].split(delimiter)]

    header_row_idx = None
    if data_start_idx > 0:
        previous_tokens = [token.strip() for token in lines[data_start_idx - 1].split(delimiter)]
        if len(previous_tokens) == len(data_tokens) and looks_like_header(previous_tokens):
            header_row_idx = data_start_idx - 1
    if header_row_idx is None and looks_like_header(data_tokens):
        header_row_idx = data_start_idx

    parse_start_idx = header_row_idx if header_row_idx is not None else data_start_idx
    parse_text = "\n".join(lines[parse_start_idx:])
    read_kwargs = {"sep": delimiter, "engine": "python"}
    if header_row_idx is None:
        read_kwargs["header"] = None

    df = pd.read_csv(io.StringIO(parse_text), **read_kwargs)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if df.empty:
        raise ValueError(f"{filename} did not produce a usable table after parsing.")

    if header_row_idx is None:
        df.columns = make_unique_columns([f"col_{idx}" for idx in range(df.shape[1])])
    else:
        df.columns = make_unique_columns(df.columns.tolist())

    column_info = detect_columns(df)
    fs_estimate_hz = DEFAULT_FS_FALLBACK
    fs_source = f"fallback default ({DEFAULT_FS_FALLBACK:.0f} Hz)"
    time_axis_sec = np.arange(len(df), dtype=float) / fs_estimate_hz

    if column_info["time_candidates"]:
        primary_time_column = column_info["time_candidates"][0]
        inferred_fs, inferred_time_axis_sec, inferred_source = estimate_fs_from_time_series(df[primary_time_column])
        if inferred_fs is not None:
            fs_estimate_hz = inferred_fs
            time_axis_sec = inferred_time_axis_sec
            fs_source = inferred_source

    marker_column = column_info["marker_column"]
    unique_markers = []
    if marker_column is not None:
        marker_series = df[marker_column].dropna().astype(str).str.strip()
        marker_series = marker_series[marker_series != ""]
        unique_markers = marker_series.unique().tolist()[:20]

    return {
        "filename": filename,
        "df": df,
        "family_guess": infer_session_family(filename),
        "delimiter": delimiter,
        "header_detected": header_row_idx is not None,
        "signal_columns": column_info["signal_candidates"],
        "marker_column": marker_column,
        "time_column": column_info["time_candidates"][0] if column_info["time_candidates"] else None,
        "fs_estimate_hz": float(fs_estimate_hz),
        "fs_source": fs_source,
        "time_axis_sec": np.asarray(time_axis_sec, dtype=float),
        "duration_sec": float(time_axis_sec[-1]) if len(time_axis_sec) else 0.0,
        "unique_markers": unique_markers,
    }


SESSIONS = {}
FAILED_CSVS = {}
for filename in SELECTED_CSV_FILES:
    try:
        SESSIONS[filename] = load_openbci_csv(UPLOADED_FILES[filename], filename)
    except Exception as exc:
        FAILED_CSVS[filename] = str(exc)

if not SESSIONS:
    raise RuntimeError("None of the selected CSV files could be parsed successfully.")

if FAILED_CSVS:
    print("Some selected CSV files could not be parsed automatically:")
    display(pd.DataFrame([{"Filename": name, "Error": error} for name, error in FAILED_CSVS.items()]))

summary_rows = []
for filename, session in SESSIONS.items():
    summary_rows.append(
        {
            "Filename": filename,
            "Rows": len(session["df"]),
            "Duration (s)": round(session["duration_sec"], 2),
            "Signal Columns": ", ".join(session["signal_columns"]) if session["signal_columns"] else "None detected",
            "Marker Column": session["marker_column"] or "None detected",
            "Unique Markers": ", ".join(map(str, session["unique_markers"][:10])) if session["unique_markers"] else "None detected",
        }
    )

FILE_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values("Filename").reset_index(drop=True)
display(FILE_SUMMARY_DF)


## Section 4 — Marker and Label Handling

Jaw labels can arrive in two ways:

### A. Marker-based labels
If a marker column exists, the notebook will use a configurable marker map to identify jaw-active intervals.

### B. Filename or protocol fallback
If markers are missing or ambiguous, the notebook falls back to a user-editable per-file role:

- `jaw`
- `rest`
- `mixed`

This makes the analysis resilient to imperfect labeling while keeping uncertainty visible instead of hiding it.


In [ ]:
MARKER_MAP = {
    "jaw_start": [1, 3],
    "jaw_stop": [2, 4],
}


def guess_file_role(filename):
    upper_name = filename.upper()
    if any(token in upper_name for token in ["JAW", "CLENCH", "HR"]):
        return "jaw"
    if "REST" in upper_name or "BASELINE" in upper_name:
        return "rest"
    return "mixed"


FILE_LABEL_OVERRIDES = {filename: guess_file_role(filename) for filename in SESSIONS}
print("Edit FILE_LABEL_OVERRIDES and rerun this cell if you want to change fallback file roles.")
display(pd.DataFrame({"Filename": list(FILE_LABEL_OVERRIDES), "Fallback Role": list(FILE_LABEL_OVERRIDES.values())}))


def normalize_marker_token(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    try:
        numeric = float(text)
        if np.isfinite(numeric):
            if abs(numeric - round(numeric)) < 1e-9:
                return int(round(numeric))
            return numeric
    except ValueError:
        pass
    return text.upper()


def extract_marker_events(session):
    marker_column = session["marker_column"]
    if marker_column is None:
        return pd.DataFrame(columns=["time_sec", "marker"])

    raw_values = session["df"][marker_column].tolist()
    time_axis = session["time_axis_sec"]
    events = []
    previous = object()
    for time_sec, raw_value in zip(time_axis, raw_values):
        marker = normalize_marker_token(raw_value)
        if marker in [None, 0, 0.0, "0", "0.0"]:
            previous = marker
            continue
        if marker == previous:
            continue
        events.append({"time_sec": float(time_sec), "marker": marker})
        previous = marker
    return pd.DataFrame(events)


def marker_matches(marker_value, allowed_values):
    for allowed in allowed_values:
        if isinstance(allowed, str):
            if str(marker_value).upper() == allowed.upper():
                return True
        else:
            try:
                if float(marker_value) == float(allowed):
                    return True
            except Exception:
                continue
    return False


def extract_jaw_intervals(session, marker_map):
    events_df = extract_marker_events(session)
    intervals = []
    active_start = None

    for row in events_df.itertuples():
        marker = row.marker
        if marker_matches(marker, marker_map["jaw_start"]):
            if active_start is None:
                active_start = float(row.time_sec)
        elif marker_matches(marker, marker_map["jaw_stop"]):
            if active_start is not None and row.time_sec > active_start:
                intervals.append({
                    "start_sec": active_start,
                    "end_sec": float(row.time_sec),
                    "source": "marker",
                })
                active_start = None

    if active_start is not None:
        intervals.append({
            "start_sec": active_start,
            "end_sec": float(session["time_axis_sec"][-1]),
            "source": "marker_open_end",
        })

    if intervals:
        intervals_df = pd.DataFrame(intervals)
        intervals_df["duration_sec"] = intervals_df["end_sec"] - intervals_df["start_sec"]
        return intervals_df

    return pd.DataFrame(columns=["start_sec", "end_sec", "source", "duration_sec"])


LABEL_SUMMARY_ROWS = []
for filename, session in SESSIONS.items():
    events_df = extract_marker_events(session)
    intervals_df = extract_jaw_intervals(session, MARKER_MAP)
    fallback_role = FILE_LABEL_OVERRIDES[filename]

    session["marker_events"] = events_df
    session["jaw_intervals"] = intervals_df
    session["fallback_role"] = fallback_role

    LABEL_SUMMARY_ROWS.append(
        {
            "Filename": filename,
            "Marker Events": len(events_df),
            "Jaw Intervals": len(intervals_df),
            "Fallback Role": fallback_role,
            "Label Source": "marker" if len(intervals_df) > 0 else f"fallback ({fallback_role})",
        }
    )

LABEL_SUMMARY_DF = pd.DataFrame(LABEL_SUMMARY_ROWS).sort_values("Filename").reset_index(drop=True)
display(LABEL_SUMMARY_DF)


## Section 5 — Preprocessing

Jaw decoding is treated here as an EMG-like control problem. The default preprocessing stack is:

1. common-reference subtraction across channels (median reference)
2. 60 Hz notch filtering to reduce line noise
3. EMG-friendly bandpass filtering (**20–100 Hz**)

An EEG-style **1–40 Hz** option is also available for comparison, but jaw decoding defaults to the EMG-focused pipeline because the practical goal is a robust click signal.


In [ ]:
def safe_filtfilt(b, a, values):
    values = np.asarray(values, dtype=float)
    if len(values) < max(len(a), len(b)) * 3:
        return values.copy()
    finite_values = values.copy()
    if np.isnan(finite_values).any():
        fill_value = np.nanmedian(finite_values)
        if not np.isfinite(fill_value):
            fill_value = 0.0
        finite_values = np.nan_to_num(finite_values, nan=fill_value)
    return filtfilt(b, a, finite_values)


def common_reference(signal_df):
    centered = signal_df.apply(clean_numeric)
    row_reference = centered.median(axis=1)
    return centered.sub(row_reference, axis=0)


def notch_filter(values, fs_hz, notch_hz=60.0, quality_factor=30.0):
    nyquist = 0.5 * fs_hz
    if notch_hz >= nyquist:
        return np.asarray(values, dtype=float)
    b, a = iirnotch(notch_hz, quality_factor, fs_hz)
    return safe_filtfilt(b, a, values)


def bandpass_filter(values, fs_hz, low_hz=20.0, high_hz=100.0, order=4):
    nyquist = 0.5 * fs_hz
    high_hz = min(high_hz, nyquist * 0.95)
    if high_hz <= low_hz:
        return np.asarray(values, dtype=float)
    b, a = butter(order, [low_hz / nyquist, high_hz / nyquist], btype="band")
    return safe_filtfilt(b, a, values)


def preprocess_session(session, mode="emg"):
    signal_columns = session["signal_columns"]
    if not signal_columns:
        raise ValueError(f"{session['filename']} has no detected signal columns.")

    fs_hz = float(session["fs_estimate_hz"])
    signal_df = session["df"].loc[:, signal_columns].apply(clean_numeric)
    referenced = common_reference(signal_df)

    if mode == "emg":
        low_hz, high_hz = 20.0, 100.0
    elif mode == "eeg":
        low_hz, high_hz = 1.0, 40.0
    else:
        raise ValueError("mode must be 'emg' or 'eeg'")

    filtered = pd.DataFrame(index=referenced.index)
    for column in signal_columns:
        values = referenced[column].to_numpy(dtype=float)
        values = notch_filter(values, fs_hz)
        values = bandpass_filter(values, fs_hz, low_hz=low_hz, high_hz=high_hz)
        filtered[column] = values

    return filtered


for session in SESSIONS.values():
    session["preprocessed_emg"] = preprocess_session(session, mode="emg")
    session["preprocessed_eeg"] = preprocess_session(session, mode="eeg")

PLOT_FILENAME = next(iter(SESSIONS))
PLOT_SESSION = SESSIONS[PLOT_FILENAME]
PLOT_CHANNELS = PLOT_SESSION["signal_columns"][: min(3, len(PLOT_SESSION["signal_columns"]))]

if PLOT_CHANNELS:
    time_axis = PLOT_SESSION["time_axis_sec"]
    zoom_mask = time_axis <= min(8.0, time_axis[-1])

    fig, axes = plt.subplots(len(PLOT_CHANNELS), 1, figsize=(14, 3 * len(PLOT_CHANNELS)), sharex=True)
    if len(PLOT_CHANNELS) == 1:
        axes = [axes]

    for ax, channel in zip(axes, PLOT_CHANNELS):
        raw_values = clean_numeric(PLOT_SESSION["df"][channel]).to_numpy(dtype=float)
        emg_values = PLOT_SESSION["preprocessed_emg"][channel].to_numpy(dtype=float)
        ax.plot(time_axis[zoom_mask], raw_values[zoom_mask], label="Raw", linewidth=1.0, alpha=0.8)
        ax.plot(time_axis[zoom_mask], emg_values[zoom_mask], label="EMG preprocessing (20–100 Hz)", linewidth=1.1)
        ax.set_title(f"{PLOT_FILENAME} — {channel}")
        ax.set_ylabel("Amplitude")
        ax.legend(loc="upper right")

    axes[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()
else:
    print(f"{PLOT_FILENAME} does not have enough detected signal channels for a before/after plot.")


## Section 6 — Window Extraction

The decoder operates on short signal windows rather than entire sessions. By default, this notebook uses:

- **1.0 second** windows
- **50% overlap**
- detected sampling rate when available, otherwise **250 Hz**

Each window receives one of three labels:

- `CLENCH`
- `REST`
- `UNKNOWN`

`UNKNOWN` windows are excluded from model training but kept available for visualization and replay.


In [ ]:
WINDOW_SECONDS = 1.0
WINDOW_OVERLAP = 0.5
WINDOW_OVERLAP_FRACTION = float(WINDOW_OVERLAP)


def overlap_duration(start_a, end_a, start_b, end_b):
    return max(0.0, min(end_a, end_b) - max(start_a, start_b))


def label_window(session, start_sec, end_sec):
    intervals_df = session["jaw_intervals"]
    fallback_role = session["fallback_role"]

    if not intervals_df.empty:
        overlap = 0.0
        for row in intervals_df.itertuples():
            overlap += overlap_duration(start_sec, end_sec, row.start_sec, row.end_sec)
        window_duration = max(end_sec - start_sec, 1e-6)
        overlap_fraction = overlap / window_duration
        return "CLENCH" if overlap_fraction >= 0.25 else "REST"

    if fallback_role == "jaw":
        return "CLENCH"
    if fallback_role == "rest":
        return "REST"
    return "UNKNOWN"


def build_windows_for_session(session, preprocessed_key="preprocessed_emg", window_seconds=1.0, overlap_fraction=0.5):
    signal_df = session[preprocessed_key]
    fs_hz = float(session["fs_estimate_hz"])
    window_samples = max(8, int(round(window_seconds * fs_hz)))
    step_samples = max(1, int(round(window_samples * (1.0 - overlap_fraction))))
    time_axis = session["time_axis_sec"]

    rows = []
    for start_idx in range(0, max(len(signal_df) - window_samples + 1, 0), step_samples):
        end_idx = start_idx + window_samples
        window_matrix = signal_df.iloc[start_idx:end_idx].to_numpy(dtype=float)
        start_sec = float(time_axis[start_idx])
        end_sec = float(time_axis[end_idx - 1])
        center_sec = 0.5 * (start_sec + end_sec)
        rows.append(
            {
                "filename": session["filename"],
                "start_idx": start_idx,
                "end_idx": end_idx,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "center_sec": center_sec,
                "label": label_window(session, start_sec, end_sec),
                "window_matrix": window_matrix,
                "fs_hz": fs_hz,
                "channels": tuple(signal_df.columns),
            }
        )

    return pd.DataFrame(rows)


WINDOWS_BY_FILE = {}
window_tables = []
for filename, session in SESSIONS.items():
    window_df = build_windows_for_session(session)
    WINDOWS_BY_FILE[filename] = window_df
    window_tables.append(window_df)

ALL_WINDOWS_DF = pd.concat(window_tables, ignore_index=True) if window_tables else pd.DataFrame()

WINDOW_CLASS_COUNTS = (
    ALL_WINDOWS_DF.groupby(["filename", "label"]).size().unstack(fill_value=0).reset_index()
    if not ALL_WINDOWS_DF.empty else pd.DataFrame()
)
display(WINDOW_CLASS_COUNTS)


## Section 7 — Feature Extraction

Each window is converted into an interpretable feature vector. The feature set is intentionally simple and explainable:

- RMS
- variance
- mean absolute value
- waveform length
- peak-to-peak range
- bandpower summaries

This makes the decoder easier to justify academically than a black-box-only approach.


In [ ]:
def bandpower(values, fs_hz, low_hz, high_hz):
    values = np.asarray(values, dtype=float)
    if len(values) < 8:
        return 0.0
    nperseg = min(len(values), max(32, int(fs_hz)))
    freqs, power = welch(values, fs=fs_hz, nperseg=nperseg)
    mask = (freqs >= low_hz) & (freqs <= high_hz)
    if not np.any(mask):
        return 0.0
    return float(np.trapz(power[mask], freqs[mask]))


def compute_window_features(window_matrix, fs_hz, channel_names):
    feature_row = {}
    window_matrix = np.asarray(window_matrix, dtype=float)

    for channel_idx, _channel_name in enumerate(channel_names):
        channel_name = f"Channel_{channel_idx + 1}"
        values = window_matrix[:, channel_idx]
        values = np.nan_to_num(values, nan=np.nanmedian(values) if np.isfinite(np.nanmedian(values)) else 0.0)
        centered = values - np.median(values)
        diffs = np.diff(centered)

        feature_row[f"{channel_name}__rms"] = float(np.sqrt(np.mean(centered ** 2)))
        feature_row[f"{channel_name}__variance"] = float(np.var(centered))
        feature_row[f"{channel_name}__mav"] = float(np.mean(np.abs(centered)))
        feature_row[f"{channel_name}__waveform_length"] = float(np.sum(np.abs(diffs)))
        feature_row[f"{channel_name}__ptp"] = float(np.ptp(centered))
        feature_row[f"{channel_name}__bandpower_20_40"] = bandpower(centered, fs_hz, 20.0, 40.0)
        feature_row[f"{channel_name}__bandpower_40_100"] = bandpower(centered, fs_hz, 40.0, 100.0)

    return feature_row


feature_rows = []
for row in ALL_WINDOWS_DF.itertuples():
    features = compute_window_features(row.window_matrix, row.fs_hz, row.channels)
    features.update(
        {
            "filename": row.filename,
            "start_sec": row.start_sec,
            "end_sec": row.end_sec,
            "center_sec": row.center_sec,
            "label": row.label,
        }
    )
    feature_rows.append(features)

FEATURE_TABLE_DF = pd.DataFrame(feature_rows).fillna(0.0)
TRAINING_FEATURE_DF = FEATURE_TABLE_DF[FEATURE_TABLE_DF["label"].isin(CLASS_ORDER)].reset_index(drop=True)

print("Window counts by class:")
display(FEATURE_TABLE_DF["label"].value_counts(dropna=False).rename_axis("Label").reset_index(name="Count"))

print("Training class balance:")
display(TRAINING_FEATURE_DF["label"].value_counts().rename_axis("Label").reset_index(name="Count"))

print("Feature preview:")
display(TRAINING_FEATURE_DF.head())


## Section 8 — Model Training

The decoder compares three interpretable baseline models:

- Logistic Regression
- Random Forest
- Linear Discriminant Analysis

When multiple sessions are available, the notebook prefers **held-out-file evaluation**. Otherwise it falls back to a stratified within-file split.


In [ ]:
if TRAINING_FEATURE_DF.empty:
    raise RuntimeError("No labeled CLENCH/REST windows were available for training. Check markers or FILE_LABEL_OVERRIDES.")

if set(TRAINING_FEATURE_DF["label"]) != set(CLASS_ORDER):
    raise RuntimeError("Training data does not contain both REST and CLENCH labels. Add more files or adjust labeling.")

FEATURE_COLUMNS = [column for column in TRAINING_FEATURE_DF.columns if "__" in column]

MODEL_SPECS = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
        ]
    ),
    "Random Forest": Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    random_state=42,
                ),
            )
        ]
    ),
    "Linear Discriminant Analysis": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LinearDiscriminantAnalysis()),
        ]
    ),
}


def predict_clench_probability(fitted_model, x_frame):
    classes = list(fitted_model.classes_)
    clench_index = classes.index("CLENCH")
    return fitted_model.predict_proba(x_frame)[:, clench_index]


def compute_metric_bundle(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=CLASS_ORDER,
        zero_division=0,
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_rest": precision[0],
        "recall_rest": recall[0],
        "f1_rest": f1[0],
        "support_rest": support[0],
        "precision_clench": precision[1],
        "recall_clench": recall[1],
        "f1_clench": f1[1],
        "support_clench": support[1],
    }
    return metrics


def evaluate_model_within_split(model_name, estimator, feature_df):
    x = feature_df[FEATURE_COLUMNS]
    y = feature_df["label"]
    train_x, test_x, train_y, test_y, train_meta, test_meta = train_test_split(
        x,
        y,
        feature_df[["filename", "center_sec"]],
        test_size=0.25,
        random_state=42,
        stratify=y,
    )
    fitted = clone(estimator)
    fitted.fit(train_x, train_y)
    pred_y = fitted.predict(test_x)
    prob_y = predict_clench_probability(fitted, test_x)
    metrics = compute_metric_bundle(test_y, pred_y)

    prediction_df = test_meta.copy().reset_index(drop=True)
    prediction_df["true_label"] = test_y.reset_index(drop=True)
    prediction_df["pred_label"] = pd.Series(pred_y)
    prediction_df["p_clench"] = pd.Series(prob_y)

    return {
        "model_name": model_name,
        "protocol": "within_file_split",
        "metrics": metrics,
        "prediction_df": prediction_df,
        "fold_summary_df": pd.DataFrame(
            [{"Fold": "random_split", "Test File": "mixed", "Samples": len(test_y), "Clench F1": metrics["f1_clench"]}]
        ),
    }


def evaluate_model_holdout_by_file(model_name, estimator, feature_df):
    prediction_rows = []
    fold_rows = []
    unique_files = sorted(feature_df["filename"].unique())

    for test_file in unique_files:
        train_mask = feature_df["filename"] != test_file
        test_mask = feature_df["filename"] == test_file

        train_df = feature_df.loc[train_mask].reset_index(drop=True)
        test_df = feature_df.loc[test_mask].reset_index(drop=True)
        if train_df.empty or test_df.empty:
            continue
        if set(train_df["label"]) != set(CLASS_ORDER):
            continue

        fitted = clone(estimator)
        fitted.fit(train_df[FEATURE_COLUMNS], train_df["label"])
        pred_y = fitted.predict(test_df[FEATURE_COLUMNS])
        prob_y = predict_clench_probability(fitted, test_df[FEATURE_COLUMNS])
        test_metrics = compute_metric_bundle(test_df["label"], pred_y)

        fold_rows.append(
            {
                "Fold": f"holdout_{test_file}",
                "Test File": test_file,
                "Samples": len(test_df),
                "Clench F1": test_metrics["f1_clench"],
            }
        )

        fold_predictions = test_df[["filename", "center_sec", "label"]].copy()
        fold_predictions = fold_predictions.rename(columns={"label": "true_label"})
        fold_predictions["pred_label"] = pred_y
        fold_predictions["p_clench"] = prob_y
        prediction_rows.append(fold_predictions)

    if not prediction_rows:
        return None

    prediction_df = pd.concat(prediction_rows, ignore_index=True)
    metrics = compute_metric_bundle(prediction_df["true_label"], prediction_df["pred_label"])
    return {
        "model_name": model_name,
        "protocol": "held_out_file",
        "metrics": metrics,
        "prediction_df": prediction_df,
        "fold_summary_df": pd.DataFrame(fold_rows),
    }


RESULTS = []
multi_file_available = TRAINING_FEATURE_DF["filename"].nunique() >= 2
for model_name, estimator in MODEL_SPECS.items():
    if multi_file_available:
        result = evaluate_model_holdout_by_file(model_name, estimator, TRAINING_FEATURE_DF)
        if result is None:
            result = evaluate_model_within_split(model_name, estimator, TRAINING_FEATURE_DF)
    else:
        result = evaluate_model_within_split(model_name, estimator, TRAINING_FEATURE_DF)
    RESULTS.append(result)

MODEL_COMPARISON_DF = pd.DataFrame(
    [
        {
            "Model": result["model_name"],
            "Protocol": result["protocol"],
            "Accuracy": result["metrics"]["accuracy"],
            "CLENCH Precision": result["metrics"]["precision_clench"],
            "CLENCH Recall": result["metrics"]["recall_clench"],
            "CLENCH F1": result["metrics"]["f1_clench"],
        }
        for result in RESULTS
    ]
).sort_values(["CLENCH F1", "CLENCH Precision", "Accuracy"], ascending=False).reset_index(drop=True)

display(MODEL_COMPARISON_DF)

BEST_MODEL_NAME = MODEL_COMPARISON_DF.iloc[0]["Model"]
BEST_RESULT = next(result for result in RESULTS if result["model_name"] == BEST_MODEL_NAME)
FINAL_MODEL = clone(MODEL_SPECS[BEST_MODEL_NAME]).fit(TRAINING_FEATURE_DF[FEATURE_COLUMNS], TRAINING_FEATURE_DF["label"])

print(f"Selected model: {BEST_MODEL_NAME} ({BEST_RESULT['protocol']})")


## Section 9 — Evaluation

The most important evaluation question is not just whether the classifier is accurate overall, but whether it is **precise and sensitive enough for CLENCH**. In a real-time system, false clicks matter.


In [ ]:
BEST_PREDICTIONS_DF = BEST_RESULT["prediction_df"].copy()

print("Fold summary for the selected model:")
display(BEST_RESULT["fold_summary_df"])

report_dict = classification_report(
    BEST_PREDICTIONS_DF["true_label"],
    BEST_PREDICTIONS_DF["pred_label"],
    labels=CLASS_ORDER,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T
display(report_df)

cm = confusion_matrix(BEST_PREDICTIONS_DF["true_label"], BEST_PREDICTIONS_DF["pred_label"], labels=CLASS_ORDER)
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER).plot(ax=ax, colorbar=False)
ax.set_title(f"Confusion Matrix — {BEST_MODEL_NAME}")
plt.show()

metrics = BEST_RESULT["metrics"]
interpretation = f"""
### Evaluation Interpretation

- **Selected model:** {BEST_MODEL_NAME}
- **Evaluation protocol:** {BEST_RESULT['protocol']}
- **Accuracy:** {metrics['accuracy']:.3f}
- **CLENCH precision:** {metrics['precision_clench']:.3f}
- **CLENCH recall:** {metrics['recall_clench']:.3f}
- **CLENCH F1:** {metrics['f1_clench']:.3f}

For a click decoder, CLENCH precision is especially important because false positives translate into unwanted clicks. CLENCH recall matters because missed clenches reduce usability.
"""
display(Markdown(interpretation))


## Section 10 — Probability Timeline

A useful real-time decoder needs good temporal behavior, not only good summary metrics. The plot below runs the selected model over all windows from one analysis file and shows the probability of `CLENCH` over time.


In [ ]:
TIMELINE_FILENAME = next(iter(SESSIONS))
TIMELINE_WINDOWS_DF = WINDOWS_BY_FILE[TIMELINE_FILENAME].copy()
TIMELINE_FEATURE_ROWS = []

for row in TIMELINE_WINDOWS_DF.itertuples():
    feature_row = compute_window_features(row.window_matrix, row.fs_hz, row.channels)
    TIMELINE_FEATURE_ROWS.append(feature_row)

TIMELINE_FEATURE_DF = pd.DataFrame(TIMELINE_FEATURE_ROWS).reindex(columns=FEATURE_COLUMNS, fill_value=0.0)
TIMELINE_WINDOWS_DF["p_clench"] = predict_clench_probability(FINAL_MODEL, TIMELINE_FEATURE_DF[FEATURE_COLUMNS])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(TIMELINE_WINDOWS_DF["center_sec"], TIMELINE_WINDOWS_DF["p_clench"], color="#2563eb", linewidth=1.7, label="P(CLENCH)")

intervals_df = SESSIONS[TIMELINE_FILENAME]["jaw_intervals"]
if not intervals_df.empty:
    for idx, row in enumerate(intervals_df.itertuples()):
        label = "Jaw marker interval" if idx == 0 else None
        ax.axvspan(row.start_sec, row.end_sec, color="#fca5a5", alpha=0.25, label=label)

ax.set_title(f"Probability Timeline — {TIMELINE_FILENAME}")
ax.set_xlabel("Time (s)")
ax.set_ylabel("P(CLENCH)")
ax.set_ylim(-0.02, 1.02)
ax.legend(loc="upper right")
plt.show()


## Section 11 — Click Trigger Replay

The notebook now converts classifier confidence into a discrete click stream. The trigger logic uses a probability threshold, rearm threshold, and cooldown to suppress repeated clicks while a single clench is being held.


In [ ]:
clench_probability_threshold = 0.80
rearm_threshold = 0.55
cooldown_ms = 300
minimum_separation_ms = 300
hold_suppression = True


def replay_clicks(probability_df, threshold=0.80, rearm=0.55, cooldown_ms=300, minimum_separation_ms=300, hold_suppression=True):
    clicks = []
    armed = True
    last_click_ms = -np.inf

    for row in probability_df.itertuples():
        time_ms = float(row.center_sec) * 1000.0
        prob = float(row.p_clench)

        if not armed:
            if prob <= rearm and (time_ms - last_click_ms) >= cooldown_ms:
                armed = True

        ready_for_click = armed and prob >= threshold and (time_ms - last_click_ms) >= minimum_separation_ms
        if ready_for_click:
            clicks.append({"time_sec": float(row.center_sec), "p_clench": prob})
            last_click_ms = time_ms
            if hold_suppression:
                armed = False

    return pd.DataFrame(clicks)


def compare_clicks_to_markers(click_df, interval_df, tolerance_sec=0.30):
    if interval_df.empty:
        return None

    matched_intervals = set()
    false_clicks = 0
    for click in click_df.itertuples():
        matched = False
        for idx, interval in interval_df.iterrows():
            if (interval["start_sec"] - tolerance_sec) <= click.time_sec <= (interval["end_sec"] + tolerance_sec):
                matched = True
                matched_intervals.add(idx)
                break
        if not matched:
            false_clicks += 1

    missed_events = len(interval_df) - len(matched_intervals)
    return {
        "detected_clicks": len(click_df),
        "false_triggers": false_clicks,
        "missed_marker_events": missed_events,
    }


CLICK_DF = replay_clicks(
    TIMELINE_WINDOWS_DF,
    threshold=clench_probability_threshold,
    rearm=rearm_threshold,
    cooldown_ms=cooldown_ms,
    minimum_separation_ms=minimum_separation_ms,
    hold_suppression=hold_suppression,
)
CLICK_SUMMARY = compare_clicks_to_markers(CLICK_DF, intervals_df)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(TIMELINE_WINDOWS_DF["center_sec"], TIMELINE_WINDOWS_DF["p_clench"], color="#1d4ed8", linewidth=1.6, label="P(CLENCH)")
ax.axhline(clench_probability_threshold, color="#dc2626", linestyle="--", linewidth=1.2, label="Click threshold")

if not intervals_df.empty:
    for idx, row in enumerate(intervals_df.itertuples()):
        label = "Marker jaw interval" if idx == 0 else None
        ax.axvspan(row.start_sec, row.end_sec, color="#fecaca", alpha=0.25, label=label)

if not CLICK_DF.empty:
    ax.scatter(CLICK_DF["time_sec"], CLICK_DF["p_clench"], color="#111827", s=50, label="Detected click")

ax.set_title(f"Click Trigger Replay — {TIMELINE_FILENAME}")
ax.set_xlabel("Time (s)")
ax.set_ylabel("P(CLENCH)")
ax.set_ylim(-0.02, 1.02)
ax.legend(loc="upper right")
plt.show()

summary_rows = [{"Detected clicks": len(CLICK_DF)}]
if CLICK_SUMMARY is not None:
    summary_rows[0]["Approx. false triggers"] = CLICK_SUMMARY["false_triggers"]
    summary_rows[0]["Approx. missed events"] = CLICK_SUMMARY["missed_marker_events"]
display(pd.DataFrame(summary_rows))


## Section 12 — Final Interpretation

The final summary below ties the decoder back to the broader hybrid BCI project.


In [ ]:
def build_final_summary(best_model_name, best_result, click_df, click_summary, timeline_filename):
    metrics = best_result["metrics"]
    false_trigger_text = "not available because no marker intervals were detected"
    missed_event_text = "not available because no marker intervals were detected"
    if click_summary is not None:
        false_trigger_text = str(click_summary["false_triggers"])
        missed_event_text = str(click_summary["missed_marker_events"])

    summary = f"""
### Final Interpretation

**Is jaw clench a strong control signal?**  
Yes. Jaw activity typically produces a larger and more repeatable biosignal than subtle left/right EEG intent, which makes it far better suited for a discrete click action.

**Why is it stronger than left/right EEG for click-style actions?**  
Jaw clench produces higher-amplitude, higher-SNR activity and is more consistent across trials. Left/right EEG intent is weaker, more variable, and more sensitive to session drift.

**How does the classifier become a real control input?**  
The model converts each window into a clench probability. A threshold-plus-rearm rule then turns that probability stream into discrete click events with cooldown protection.

**Selected model and result**  
- Model: **{best_model_name}**
- Protocol: **{best_result['protocol']}**
- CLENCH precision: **{metrics['precision_clench']:.3f}**
- CLENCH recall: **{metrics['recall_clench']:.3f}**
- CLENCH F1: **{metrics['f1_clench']:.3f}**
- Replay file: **{timeline_filename}**
- Detected clicks in replay: **{len(click_df)}**
- Approximate false triggers: **{false_trigger_text}**
- Approximate missed marker events: **{missed_event_text}**

**Limitations**  
The decoder still depends on file quality, labeling quality, and threshold selection. False positives can appear if noise or movement artifacts mimic jaw activity, and fallback labels are weaker than clean marker-based labels.

**Connection to the final hybrid BCI game**  
In the hybrid system, jaw acts as the reliable discrete trigger while EEG contributes directional intent. That division of labor is what makes the combined interface more practical than EEG-only control.
"""
    display(Markdown(summary))


build_final_summary(BEST_MODEL_NAME, BEST_RESULT, CLICK_DF, CLICK_SUMMARY, TIMELINE_FILENAME)
